# DASHI: Dataset shift analysis and characterization in python

#### David Fernández Narro, Pablo Ferri Borredá, Ángel Sanchez-García, Juan M. García-Gómez, Carlos Sáez 

*dfernar@upv.edu.es
**pabferb2@upv.es
***ansan12a@upv.es
****juanmig@upv.es
*****carsaesi@upv.es

---
##### Library: `dashi`

Welcome to the `dashi` Python library! This notebook demonstrates the key features and functionality of the library, as well as an example to help you to understand its usage and applications for **dataset shift characterization**.


## Contents
- [1 Introduction](#introduction)
    - [1.1 What is `dashi`?](#what-is-dashi)
    - [1.2 Key Features](#key-features)
    - [1.3 Installation](#installation)
- [2 Unsupervised characterization analysis](#unsupervised)
    - [2.1 Data pre-processing](#unsupervised-preprocessing)
    - [2.2 Data analysis](#unsupervised-characterization)
    - [2.3 Results display](#unsupervised-visualization)
    - [2.4 Interpretation of temporal changes in IGT projections](#unsupervised-interpretation)
    - [2.5 Summary of dashi unsupervised characterization functions](#unsupervised-summary)
- [3 Supervised characterization analysis](#supervised)
    - [3.1 Data pre-processing](#supervised-preprocessing)
    - [3.2 Data analysis](#supervised-analysis)
    - [3.3 Results display](#supervised-visualization)
    - [3.4 Metrics arrangement](#supervised-arrangement)
    - [3.5 Interpretation of the results](#supervised-interpretation)
    - [3.6 Summary of dashi supervised characterization functions](#supervised-summary)
- [References](#references)

<a id="introduction"></a>
## 1 Introduction

---

<a id="what-is-dashi"></a>
### 1.1 What is `dashi`?

`dashi` is a Python library crafted to **analyze and characterize temporal and multi-source dataset shifts**. It equips users with powerful tools for both supervised and unsupervised evaluations, making it easier to detect, understand, and address changes in data distributions with confidence and precision.

#### Why is `dashi` important?
Dataset shifts—unexpected changes in data distribution over time or across sources—can significantly impact the performance of machine learning models. `dashi` helps users not only identify these shifts but also provides insights to mitigate their effects, ensuring robust and reliable models.
<br><br>

<a id="key-features"></a>
### 1.2 Key Features
Biomedical data repositories and proprietary biomedical research databases are expanding rapidly, both in terms of sample size and the diversity of collected variables. This 
growth is driven by the widespread adoption of data-sharing initiatives, advancements in technological infrastructures, and the continuous population of these repositories over 
extended periods (Gewin, 2016; Andreu-Perez et al., 2015).

However, this increased availability of data presents significant challenges. The integration of data from diverse sources over time introduces potential issues 
that can impede its reuse in research contexts, such as population studies or statistical and machine learning modeling. Differences in protocols, population characteristics, 
and unforeseen biases, whether introduced by systems or human error, can lead to **temporal or multi-source dataset shifts** 
(Quiñonero-Candela, 2009; Moreno-Torres et al., 2012). These shifts manifest as changes in statistical distributions, altering reference characteristics and potentially 
degrading model performance. Addressing these issues is particularly critical for ensuring robust and reliable predictive modeling and population health studies, as temporal 
shifts in electronic health records (EHRs) have been identified as a major concern (Sáez et al., 2020; Schlegel & Ficheur, 2017).

This variability underscores the importance of addressing **Data Quality (DQ)** as a critical factor in enabling the reliable reuse of biomedical data. By detecting, 
understanding, 
and mitigating dataset shifts, researchers can ensure the robustness and validity of their findings, even within the context of an evolving and diverse data landscape.

#### 1.2.1 Supervised Characterization

- Leverage **Random Forest classifiers or regressors** trained on batched data (temporal or multi-source) to analyze dataset shifts.  
- Evaluate how shifts in the data influence model performance and uncover potential areas of degradation.  
- Gain actionable insights to adapt your models to evolving data landscapes.

#### 1.2.2 Unsupervised Characterization

- Detect and interpret **temporal dataset shifts** without relying on labeled data by visualizing patterns of data variability.  
- Core capabilities include:  
  - **Estimating statistical distributions** over time, capturing the essence of data changes (Sáez et al., 2015)..  
  - **Projecting these distributions** onto non-parametric statistical manifolds to reveal hidden patterns of temporal variability (Sáez & García-Gómez, 2018)..  
  - **Visualizing latent trends** and shifts, providing a deeper understanding of how data evolves (Sáez et al., 2016)..  

With `dashi`, users can confidently address the challenges of dynamic datasets, ensuring their models remain robust in real-world applications.

<a id="installation"></a>
### 1.3 Installation
You can install `dashi` using pip:

```bash
pip install dashi
```

Or install from source:

```bash
git clone https://github.com/bdslab-upv/dashi
cd dashi
pip install .
```


In [ ]:
%pip install -U dashi

<a id="unsupervised"></a>
## 2 Unsupervised characterization analysis

<a id="unsupervised-preprocessing"></a>
### 2.1 Data pre-processing

---

#### 2.1.1 Load the CSV input file
The first step is to read the CSV file that contains the data for the analysis. To do it, the user can apply the `read_csv` function from `pandas` library. 
An example of how to read the CSV file is shown next:

In [ ]:
import pandas as pd
import plotly.io as pio

dataset = pd.read_csv('https://media.githubusercontent.com/media/bdslab-upv/dashi/refs/heads/main/examples/SAMPLE_FULLCOVIDMEXICO.csv', low_memory=False)
# data cleaning
dataset = dataset.drop(columns=['FECHA_ACTUALIZACION', 'ID_REGISTRO', 'FECHA_SINTOMAS', 'FECHA_DEF', 'YEAR'])

This dataset corresponds to Mexico's COVID-19 data from 2020 to 2024. It is a public dataset availabe at (https://www.gob.mx/salud/documentos/datos-abiertos-152127). In this example, we have selected a random subset of 500,000 patients from the original dataset. It should be mentioned that in 2024, the 'RESULTADO_LAB' variable was replaced by 2 new variables: 'RESULTADO_PCR' and 'RESULTADO_PCR_COINFECCION'. The variable 'CLASIFICACION_FINAL_FLU' was also introduced in 2024. 

#### 2.1.2 Transform the data into the correct format
The second step involves transforming the data into appropriate data types using the `format_data` function. This ensures consistency and compatibility for further analysis. This function has as input arguments:

- `input_dataframe`: `Pandas` `DataFrame` object with at least one columns of dates.
- `date_column_name`: is converted into the `pandas` `datetime` type for proper handling of temporal data
- `date_format`: Structure of date format. By default '%y/%m/%d'.
- `numerical_column_names`: A list containing all the numerical column names in the dataset. If this parameter is None, the variables types must be managed by the user. Numerical variables are transformed into Python's `float`type to standarize numerical operations
- `categorical_column_names`: A list containing all the categorical column names in the dataset. If this parameter is None, the variables types must be managed by the user. Categorical variables are converted into the `pandas` `category` type to optimize storage and improve processing efficiency

The `format_data` function returns a `pandas` `DataFrame` with the date column transformed into `date` python format, the categorical variables into `category` type and numerical variables into `float` type.

In [ ]:
import dashi as ds 

DATE_COLUMN = 'FECHA_INGRESO'
CATEGORICAL_VARIABLES = ['ORIGEN', 'SECTOR', 'ENTIDAD_UM', 'SEXO', 'ENTIDAD_NAC', 'ENTIDAD_RES',
                         'MUNICIPIO_RES', 'TIPO_PACIENTE', 'INTUBADO',
                         'NEUMONIA', 'EDAD', 'NACIONALIDAD', 'EMBARAZO', 'HABLA_LENGUA_INDIG',
                         'INDIGENA', 'DIABETES', 'EPOC', 'ASMA', 'INMUSUPR', 'HIPERTENSION',
                         'OTRA_COM', 'CARDIOVASCULAR', 'OBESIDAD', 'RENAL_CRONICA', 'TABAQUISMO',
                         'OTRO_CASO', 'TOMA_MUESTRA_LAB', 'RESULTADO_LAB',
                         'TOMA_MUESTRA_ANTIGENO', 'RESULTADO_ANTIGENO',
                         'CLASIFICACION_FINAL_COVID', 'MIGRANTE', 'PAIS_NACIONALIDAD',
                         'PAIS_ORIGEN', 'UCI', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION',
                         'CLASIFICACION_FINAL_FLU']

dataset_formatted = ds.format_data(input_dataframe=dataset, date_column_name=DATE_COLUMN, date_format='%Y-%m-%d',
                                categorical_column_names=CATEGORICAL_VARIABLES)

<a id="unsupervised-analysis"></a>
### 2.2 Data analysis

---

#### 2.2.1 Estimate univariate data temporal maps

The `estimate_univariate_data_temporal_map` function estimates a `DataTemporalMap` object from a `DataFrame` containig individuals in rows and the variables in columns, being one of these columns the analysis date in `date` format. This function has as input the following arguments:

- `data`: A DataFrame containing as many rows as individuals, and as many columns as teh analysis variables plus the individual acquisition date.
- `date_column_name`: A string indicating teh name of the column in data containing the analysis date variable.
- `period`: The period to batch the data for analysis. Options are: - 'week' (weekly analysis) - 'month' (monthly analysis, default) - 'year' (annual analysis)
- `start_date`: A `date` object indicating the date at which to start teh analysis, in case of being different from the first chronological date in the date column.
- `end_date`: A date object indicating the date at which to end the analysis, in case of being different from the last chronological date in the date column.
- `supports`: A dictionary with structure {variable_name: variable_type_name} containing the support of the data distributions for each variable. If not provided, it is automatically estimated from the data.
- `numeric_variables_bins`: The number of bins at which to define the frequency/ density histogram for numerical variables when their support is not provided. 100 as default.
- `numeric_smoothing`: Logical value indicating whether a Kernel Density Estimation smoothing (Gaussian kernel, default bandwidth) is to be applied on numerical variables or traditional histogram instead.
- `date_gaps_smoothing`: Logical value indicating whether a linear smoothing is applied to those time batches without data. By default gaps are filled with NAs.
- `verbose`: Whether to display additional information during the process. Defaults to False.

The `estimate_univariate_data_temporal_map` returns a `DataTemporalMap` object or a dictionary of `DataTemporalMap` objects depending on the number of analysis variables.

In [ ]:
univariate_dtm = ds.estimate_univariate_data_temporal_map(data=dataset_formatted, 
                                                       date_column_name='FECHA_INGRESO', 
                                                       period='month',
                                                       start_date=pd.to_datetime('2020-04-01'), 
                                                       end_date=pd.to_datetime('2023-12-31'),
                                                       numeric_smoothing=True,
                                                       verbose=True)

#### 2.2.2 Estimate univariate data source maps
The `estimate_univariate_data_source_map` function estimates a `DataSourceMap` object from a `DataFrame` containing individuals in rows and the variables in columns, one of which is the data‐source identifier. This function has as input the following arguments:

- `data`: A DataFrame with as many rows as individuals and as many columns as analysis variables plus the source column.

- `source_column`: Name of the column in data that contains the data‐source label; this column will be used to group and compare distributions.

- `supports`: A dictionary of the form `{variable_name: variable_type_name}` specifying the support of each variable’s distribution. If omitted, supports are estimated automatically from the data.

- `numeric_smoothing`: Whether to apply Kernel Density Estimation (Gaussian kernel, default bandwidth) on numeric variables instead of a traditional histogram. Defaults to `False`.

- `numeric_variables_bins`: Number of bins for frequency/density histograms of numeric variables when no support is provided. Defaults to 100.

- `verbose`: If True, prints progress and diagnostic information during estimation. Defaults to `False`.

The `estimate_univariate_data_source_map` returns a `DataSourceMap` object, or, if you supplied multiple analysis variables, a dictionary of `DataSourceMap` objects keyed by variable name.

In [ ]:
univariate_dsm = ds.estimate_univariate_data_source_map(
    data=dataset_formatted,
    source_column='ORIGEN',
    numeric_smoothing=True,
    verbose=True
)

#### 2.2.3 Estimate multivariate data temporal maps

The `estimate_multivariate_data_temporal_map` function estimates a `MultivariateDataTemporalMap` object from a `DataFrame` containing individuals in rows and the variables in columns, being one of these columns the analysis date in `date`format. This function allows to do a multivariate temporal variability study in 2, or 3 dimensions by applying a dimensionality reduction method to the original data. This function has as input the following arguments:

- `data`: A DataFrame where each row represents an individual or data point, and each column represents a variable. One column should represent the analysis date (typically the acquisition date). Note: in case of performing a covariate analysis, the input data should not have the label variable. 
- `date_column_name`: A string indicating the name of the column in data containing the analysis date variable.
- `kde_resolution`: The resolution of the grid used for Kernel Density Estimation (KDE). This determines the granularity of the KDE grid and how fine or coarse the estimated density maps will be. Default is 10.
- `dimensions`: The number of dimensions to keep after applying dimensionality reduction (e. g., PCA). Default is 2, meaning the data will be projected into a 2D space. The maximum number of dimensions available are 3.
- `period`: The period to batch the data for analysis. Options are: - 'week' (weekly analysis) - 'month' (monthly analysis, default) - 'year' (annual analysis)
- `start_date`: A date object indicating the date at which to start teh analysis, in case of being different from the first chronological date in the date column.
- `end_date`: A date object indicating the date at which to end the analysis, in case of being different from the last chronological date in the date column.
- `dim_reduction`: A dimensionality reduction technique to be used on the data. Default is PCA (Principal Component Analysis) for numerical data. Other options can include 'MCA' (Multiple Correspondence Analysis) for categorical data or 'FAMD' (Factor Analysis of Mixed Data) for mixed data. Note: in case of using 'FAMD', numerical variables must be in float type. Otherwise they will be treated as categorical.
- `scale`: Applicable just when using PCA dimensionality reduction. If true scales the input data using z-score normalization. Defaults to `True`.
- `scatter_plot`: Whether to generate a scatter plot of the first two principal components of the dimensionality reduction
- `verbose`: Whether to display additional information during the process. Defaults to False.

The `estimate_multivariate_data_temporal_map` returns a `MultivariateDataTemporalMap` object with the result of the study.

In [ ]:
# set up plotly renders settings for ploting images
pio.renderers.default = 'notebook'

# Drop the label columns from the data to perform a covariate analysis
LABEL_NAME = 'CLASIFICACION_FINAL_COVID'
dataset_without_label = dataset_formatted.drop(columns=[LABEL_NAME])

# Perform the analysis
multivariate_dtm = ds.estimate_multivariate_data_temporal_map(data=dataset_without_label, 
                                                              date_column_name='FECHA_INGRESO',
                                                              kde_resolution=20,
                                                              dimensions=3,
                                                              period='month',
                                                              dim_reduction='MCA',
                                                              scatter_plot=True,
                                                              verbose=True)

#### 2.2.4 Estimate multivariate data source maps
The `estimate_multivariate_data_source_map` function estimates a `MultiVariateDataSourceMap` object from a `DataFrame` containing individuals in rows and variables in columns, one of which is the data‐source identifier. It applies a dimensionality reduction method (e.g., PCA, MCA, FAMD) to handle high-dimensional data and then estimates the probability distributions for each source. This function has as input the following arguments:

- `data` : A DataFrame where each row represents an individual or data point, and each column represents a variable. One column must contain the source identifier.

- `source_column_name` : Name of the column in data that contains the data‐source label. This column is used to group and compare distributions.

- `kde_resolution` : Resolution of the grid for Kernel Density Estimation (KDE). Higher values yield finer‐grained density maps. Defaults to 10.

- `dimensions` : Number of dimensions to retain after dimensionality reduction. Default is 2 (2D projection). Maximum supported is 3.

- `dim_reduction` : Dimensionality reduction technique to apply.

    - `"PCA"` (default) for numerical data
    - `"MCA"` for categorical data
    - `"FAMD"` for mixed data (numerical variables must be floats)
- `scale` : If using PCA, whether to z-score normalize input variables before reduction. Defaults to `True`.

- `scatter_plot` : If `True`, produces a scatter plot of the first two components after reduction. Defaults to `False`.

- `verbose` : If `True`, prints progress and diagnostic messages. Defaults to `False`.

The `estimate_multivariate_data_source_map` returns a `MultiVariateDataSourceMap` object containing the estimated probability distributions per source, the multivariate density maps, counts maps, and supports for each reduced dimension.

In [ ]:
multivariate_dsm = ds.estimate_multivariate_data_source_map(
    data=dataset_without_label,
    source_column_name='ORIGEN',
    dimensions=3,
    dim_reduction='MCA',
    scatter_plot=True,
    verbose=True
)

#### 2.2.5 Estimate conditional data temporal maps

The `estimate_conditional_data_temporal_map` function enables 1, 2 or 3 dimensional conditional variability studies by generating a `MultiVariateDataTemporalMap` based on the selected label column from the dataset. This function has as input the following arguments:

- `data`: A DataFrame where each row represents an individual or data point, and each column represents a variable. One column should represent the analysis date (typically the acquisition date).
- `date_column_name`: A string indicating the name of the column in data containing the analysis date variable.
- `label_column_name`: The name of the column that contains the labels or class/ category for each observation used for conditional analysis.
- `kde_resolution`: The resolution of the grid used for Kernel Density Estimation (KDE). This determines the granularity of the KDE grid and how fine or coarse the estimated density maps will be. Default is 10.
- `dimensions`: The number of dimensions to keep after applying dimensionality reduction (e. g., PCA). Default is 2, meaning the data will be projected into a 2D space. The maximum number of dimensions available are 3. For single variable datasets, dimensions can be set to 1
- `period`: The period to batch the data for analysis. Options are: - 'week' (weekly analysis) - 'month' (monthly analysis, default) - 'year' (annual analysis)
- `start_date`: A date object indicating the date at which to start teh analysis, in case of being different from the first chronological date in the date column.
- `end_date`: A date object indicating the date at which to end the analysis, in case of being different from the last chronological date in the date column.
- `dim_reduction`:  A dimensionality reduction technique to be used on the data. Default is 'PCA' (Principal Component Analysis) for numerical data. Other options can include 'MCA' (Multiple Correspondence Analysis) for categorical data or 'FAMD' (Factor Analysis of Mixed Data) for mixed data. Note: in case of using 'FAMD', numerical variables must be in float type. Otherwise they will be treated as categorical.
- `scale`: Applicable just when using PCA dimensionality reduction. If true scales the input data using z-score normalization. Defaults to `True`.
- `scatter_plot`: Whether to generate a scatter plot of the first two principal components of the dimensionality reduction.
- `verbose`: Whether to display additional information during the process. Defaults to False.

The `estimate_conditional_data_temporal_map` returns a dictionary where the keys are the values of the selected label, and the values are the `MultiVariateDataTemporalMap` objects generated for each label. 

In [ ]:
conditional_dtm = ds.estimate_conditional_data_temporal_map(data=dataset_formatted,
                                                            date_column_name='FECHA_INGRESO',
                                                            label_column_name='CLASIFICACION_FINAL_COVID',
                                                            kde_resolution=20,
                                                            dimensions=3,
                                                            period='month',
                                                            dim_reduction='MCA',
                                                            scatter_plot=True,
                                                            verbose=True)          

#### 2.2.6 Estimate conditional data source maps

The `estimate_conditional_data_source_map` function generates one- to three-dimensional conditional variability studies by creating a `MultiVariateDataSourceMap` for each value of a specified label column, grouping data by source. This function has as input the following arguments:

- `data`: A DataFrame where each row represents an individual or data point, and each column represents a variable. One column must contain the data‐source identifier and another the label for conditional analysis.
- `source_column_name` : A string indicating the name of the column in data that holds the data‐source label used to group distributions.
- `label_column_name`: A string indicating the name of the column in data that holds the labels or classes for conditional analysis.
- `kde_resolution`: The resolution of the grid used for Kernel Density Estimation (KDE), determining the granularity of the density maps. Defaults to 10.
- `dimensions`: The number of dimensions to retain after applying dimensionality reduction (e.g., PCA). Defaults to 2 (2D projection); maximum is 3. For single‐variable datasets, set to 1.
`dim_reduction`: A dimensionality reduction technique to apply. Options are:
    - `'PCA'` (default) for numerical data
    - `'MCA'` for categorical data
    - `'FAMD'` for mixed data (numerical variables must be floats)
- `scale`: If using PCA, whether to z‐score normalize input variables before reduction. Defaults to `True`.
- `scatter_plot`: Whether to generate a scatter plot of the first two components after dimensionality reduction. Defaults to `False`.
- `verbose`: Whether to print progress and diagnostic messages during the process. Defaults to `False`.

The `estimate_conditional_data_source_map` function returns a `Dict[str, MultiVariateDataSourceMap]` where each key is a label value and each value is the corresponding `MultiVariateDataSourceMap` object.

In [ ]:
conditional_dsm = ds.estimate_conditional_data_source_map(
    data=dataset_formatted,
    source_column_name='ORIGEN',
    label_column_name='CLASIFICACION_FINAL_COVID',
    dimensions=3,
    dim_reduction='MCA',
    scatter_plot=True,
    verbose=True
)

#### 2.2.7 Variability metrics

**IGT projections for temporal variability**

The `estimate_igt_projection` function estimates a `IGTProjection` object from a `DataTemporalMap`, a `MultivariateDataTemporalMap` or a dictionary containing {label: `MultivariateDataTemporalMap`} objects. The IGT projection is a technique to visualize the temporal relationships between data batches by projecting the data into a lower-dimensional space (e. g., 2D or 3D), with time batches represented as points. The distance between points reflects the probabilistic distance between the data distributions of those time batches. This function has as input the following arguments:
- `data_temporal_map`: The temporal data map to project. This can either be a DataTemporalMap object (result of estimate_univariate_data_temporal_map), a MultiVariateDataTemporalMap object (result of estimate_multivariate_data_temporal_map), or a dictionary of MultiVariateDataTemporalMap objects where the keys are the selected labels (result of estimate_conditional_data_temporal_map).
- `dimensions`: The number of dimensions to use for the projection (2 or 3). Defaults to 2.
- `start_date`: The starting date for the temporal plot. If None, it is not constrained. Default is None.
- `end_date`: The ending date for the temporal plot. If None, it is not constrained. Default is None.
- `embedding_type`: The type of embedding technique to use for dimensionality reduction. Choices are 'classicalmds' (Classical Multidimensional Scaling), 'pca' (Principal Component Analysis) and 'nonmetricmds' (Non Metric Multidimensional Scaling). Defaults to 'classicalmds'.

The `estimate_igt_projection` returns an `IGTProjection` object.

In [ ]:
univariate_igt = ds.estimate_igt_projection(data_temporal_map=univariate_dtm[LABEL_NAME],
                                            dimensions=3,
                                            embedding_type='classicalmds')

multivariate_igt = ds.estimate_igt_projection(data_temporal_map=multivariate_dtm,
                                              dimensions=3,
                                              embedding_type='classicalmds')

conditional_igt = ds.estimate_igt_projection(data_temporal_map=conditional_dtm,
                                             dimensions=3,
                                             embedding_type='classicalmds')

**MSV for multi-source variability**

In multi-source biomedical data, contextual biases can introduce unexpected variability among the probability distributions of each source, potentially leading to irreproducible results. To quantify and monitor this spatial instability, Sáez et al. proposed two bounded, information-theoretic metrics based on simplicial projections of pairwise distribution distances:

- **Global Probabilistic Deviation (GPD)** Measures the overall multi-source variability by computing the normalized “standard deviation” of the simplex vertices (one per source) around their centroid. This yields a single value in [0, 1] that reflects how dispersed the sources’ distributions are.

- **Source Probabilistic Outlyingness (SPO)** For each source, quantifies its normalized distance from the latent central distribution (the simplex centroid). Values in [0, 1] indicate how “outlying” a particular source is relative to all others.

Both metrics use the square root of the Jensen–Shannon divergence to ensure robustness to multi-modal, multi-type, and multivariate data, providing a concise and interpretable assessment of data-source stability. 
These metrics can be obtained using the `estimate_MSV_metrics` function from a data source map, which can be a `DataSourceMap`, a `MultiVariateDataSourceMap`, or a dictionary of `MultiVariateDataSourceMap` objects. This enables quantification of variability across one or more data sources. This function has as input the following arguments:

- `data_source_map`: A `DataSourceMap` (from `estimate_univariate_data_source_map`), a `MultiVariateDataSourceMap` (from `estimate_multivariate_data_source_map`), or a `dictionary {label: MultiVariateDataSourceMap}` (from `estimate_conditional_data_source_map`).
The `estimate_MSV_metrics` function returns an `MSVMetrics` object containing the Generalized Pairwise Distance (GPD), Source Pairwise Overlap (SPO), vertices, source identifiers, and counts per source.

In [ ]:
univariate_msv = ds.estimate_MSV_metrics(
    data_source_map=univariate_dsm['CLASIFICACION_FINAL_COVID']
)
multivariate_msv = ds.estimate_MSV_metrics(
    data_source_map=multivariate_dsm
)
conditional_msv = ds.estimate_MSV_metrics(
    data_source_map=conditional_dsm
)

<a id="unsupervised-visualization"></a>
### 2.3 Results display

---

The unsupervised characterization of  the `dashi` library offers two different options for displaying the results. Data temporal maps can be represented as heatmaps, or series. IGT projections can be shown as IGT plots, depending on the desired result. 

#### 2.3.1 Plot univariate data temporal maps

The `plot_univariate_data_temporal_map` function returns a heatmap or a time series plot of a `DataTemporalMap` object, previously obtained with the `estimate_univariate_data_temporal_map` function. This function has as input the following arguments:

- `data_temporal_map`: The DataTemporalMap object that contains the temporal data to be plotted
- `absolute`: If True, plot absolute values; otherwise, the relative probabilities are plotted. Default is False.
- `log_transform`: If True, applies a log transformation to the data for better visibility of small values. Default is False.
- `start_value`: The value at which to start the plot. Default is 0.
- `end_value`: The value at which to end the plot. If None, the plot extends to the last value. Default is None.
- `start_date`: The starting date for the plot (filters the data). If None, uses the first date in the data. Default is None.
- `end_date`: The ending date for the plot (filters the data). If None, uses the last date in the data. Default is None.
- `sorting_method`: The method by which the data will be sorted for display (e. g., 'frequency', 'alphabetical'). Default is 'frequency'.
- `color_palette`: The color palette to be used for the plot (e. g., 'Spectral', 'viridis', 'viridis_r', 'magma', 'magma_r). Default is 'Spectral'.
- `mode`: The mode of visualization (e. g., 'heatmap', 'series'). Default is 'heatmap'.
- `plot_title`: The title of the plot. If None, a default title is used. Default is None.

The `plot_univariate_data_temporal_map` function returns the Plotly figure object representing the plot.

In [ ]:
univariate_dtm_heatmap = ds.plot_univariate_data_temporal_map(data_temporal_map=univariate_dtm[LABEL_NAME],
                                     absolute=False,
                                     log_transform=False,
                                     sorting_method='frequency',
                                     mode='heatmap')
univariate_dtm_heatmap.show()

#### 2.3.2 Plot univariate data source maps
The `plot_univariate_data_source_map` function visualizes a `DataSourceMap` previously obtained with the `estimate_univatiate_data_source_map` function. This function has as input the following arguments:

- `data_source_map`: A `DataSourceMap` object containing the source‐grouped probability distributions to plot.
- `absolute`: If `True`, plots absolute counts; otherwise, plots relative probabilities. Default to `False`.
- `log_transform`: If `True`, applies a log transformation to enhance visibility of small values. Default to `False`.
- `start_value`: Index or value at which to start the plot. Default to 0.
- `end_value`: Index or value at which to end the plot. If `None`, extends to the last available value. Default to `None`.
- `sorting_method`: How to order the categories on the plot (e.g., `'alphabetical', 'frequency'`). Default to `'alphabetical'`.
- `color_palette`: Name of the Plotly color palette to use (e. g., `'Spectral', 'viridis', 'viridis_r', 'magma', 'magma_r'`). Default to `'Spectral'`.
- `mode`: Type of visualization, either `'heatmap'` or `'series'`. Defaults to `'heatmap'`.
- `title`: Plot title. If `None`, a default title is generated. Defaults to `None`.
  
The `plot_univariate_data_source_map` function returns a Plotly Figure object representing the requested visualization.

In [ ]:
univariate_dsm_heatmap = ds.plot_univariate_data_source_map(
    data_source_map=univariate_dsm['CLASIFICACION_FINAL_COVID'],
    absolute=False,
    log_transform=False,
    sorting_method='alphabetical'
)
univariate_dsm_heatmap.show()

#### 2.3.3 Plot multivariate data temporal maps

The `plot_multivariate_data_temporal_map` function displays a heatmap for each of the retained dimension in the `MultivariateDataTemporalMap` object, previously obtained with the `estimate_multivariate_data_temporal_map` function. This function has as input the following arguments:

- `data_temporal_map`: The MultiVariateDataTemporalMap object that contains the temporal data to be plotted.
- `absolute`: If True, plot absolute values; otherwise, the relative probabilities are plotted. Default is False.
- `log_transform`: If True, applies a log transformation to the data for better visibility of small values. Default is False.

The `plot_multivariate_data_temporal_map` function returns the Plotly figure object representing the plot.

In [ ]:
multivariate_dtm_heatmap = ds.plot_multivariate_data_temporal_map(data_temporal_map=multivariate_dtm)
multivariate_dtm_heatmap.show()

#### 2.3.4 Plot multivariate data source maps

The `plot_multivariate_data_source_map` function plots a multivariate data source heatmap from a `MultiVariateDataSourceMap` object. This function has as input the following arguments:

- `data_source_map`: The `MultiVariateDataSourceMap` object containing multivariate probability distributions grouped by source to be visualized.

- `absolute`: If `True`, plots absolute values; otherwise, plots relative probabilities. Default to `False`.

The `plot_multivariate_data_source_map` function returns a Plotly `Figure` object representing the multivariate heatmap.

In [ ]:
multivariate_dsm_plot = ds.plot_multivariate_data_source_map(
    data_source_map=multivariate_dsm
)
multivariate_dsm_plot.show()

#### 2.3.5 Plot conditional data temporal maps

The `plot_conditional_data_temporal_map` function displays a conditioned by label heatmap for each of the retained dimensions in the object resulting from the `estimate_conditional_data_temporal_map` function. This function has as input the following arguments: 

- `data_temporal_map_dict`: A dictionary where keys are labels (strings), and values are the corresponding MultiVariateDataTemporalMap objects obtained from the 'estimate_conditional_data_temporal_map' function.
- `absolute`: If True, plot absolute values; otherwise, relative probabilities are plotted. Default is False.
- `log_transform` : If True, applies a log transformation to the data for better visibility of small values. Default is False.

The `plot_conditional_data_temporal_map` function returns a list of Plotly figure objects representing the conditional data temporal heatmaps for each dimension.

In [ ]:
conditional_dtm_heatmap = ds.plot_conditional_data_temporal_map(data_temporal_map_dict=conditional_dtm)
for fig in conditional_dtm_heatmap:
    fig.show()

#### 2.3.6 Plot conditional data source maps

The `plot_conditional_data_source_map` function displays a conditioned by label heatmap for each of the retained dimensions in the object resulting from the `estimate_conditional_data_source_map` function. This function has as input the following arguments:

- `data_source_map_dict`: A dictionary where each key is a label (str) and each value is the corresponding `MultiVariateDataSourceMap` object (from `estimate_conditional_data_source_map`).

- `absolute`: If `True`, plots absolute counts; otherwise, plots relative probabilities. Default to `False`.

The `plot_conditional_data_source_map` function returns a list of Plotly figure objects representing the conditional data source maps for each dimension.

In [ ]:
conditional_dsm_plot = ds.plot_conditional_data_source_map(
    data_source_map_dict=conditional_dsm
)
for fig in conditional_dsm_plot:
    fig.show()

#### 2.3.7 Plot IGT projections

The `plot_IGT_projection` function returns an interactive Information Geometric Temporal (IGT) plot from an `IGTProjection` object. An IGT plot visualizes the variability among time batches in a data repository in a 2D or 3D plot. Time batches are positioned as points where the distance between them represents the probabilistic distance between their distributions (currently Jensen-Shannon distance). This function has as input the following arguments:

- `igt_projection`: The IGTProjection object containing the data for the temporal plot.
- `dimensions`: The number of dimensions to be used for plotting the IGT projection (2D or 3D). Default is 2.
- `start_date`: The starting date for the temporal plot. If None, it is not constrained. Default is None.
- `end_date`: The ending date for the temporal plot. If None, it is not constrained. Default is None.
- `color_palette`: The color palette to be used for coloring the points (e. g., `'Spectral', 'viridis', 'viridis_r', 'magma', 'magma_r'`). Default is Spectral.
- `trajectory`:  If True, a smoothed trajectory showing the evolution of the information across time is plotted. Default is False.

The `plot_IGT_projection` function returns the Plotly figure object containing the IGT projection plot.

To track the temporal evolution, temporal batches are labeled to show their date and colored according to their season or period, according to the analysis period, as follows. If period=="year" the label is "yy" (2 digit year) and the color is according to year. If period=="month" the label is "yym" (yy + abbreviated month*) and the color is according to the season (yearly). If period=="week" the label is "yymmw" (yym + ISO week number in 1-2 digit) and the color is according to the season (yearly).
Note that since the projection is based on multidimensional scaling, a 2 dimensional projection entails a loss of information compared to a 3 dimensional projection. E. g., periodic variability components such as seasonal effect can be hindered by an abrupt change or a general trend.

In [ ]:
univariate_igt_plot = ds.plot_IGT_projection(igt_projection=univariate_igt,
                       dimensions=3,
                       trajectory=True)
univariate_igt_plot.show()

In [ ]:
multivariate_igt_plot = ds.plot_IGT_projection(igt_projection=multivariate_igt,
                                               dimensions=3,
                                               trajectory=True)
multivariate_igt_plot.show()

In [ ]:
conditional_igt_projection = ds.plot_IGT_projection(igt_projection=conditional_igt,
                                                    dimensions=3,
                                                    trajectory=False)
conditional_igt_projection.show()

#### 2.3.8 Plot MSV metrics

The `plot_MSV` function visualizes Multi Source Variability (MSV) metrics from an MSVMetrics object. This function has as input the following arguments:

- `msv_metrics`: An instance of the MSVMetrics class containing GPD, SPO, vertices, source identifiers, and counts by source.

- `dimensions`: Number of dimensions to plot (1, 2, or 3). Defaults to 1.

- `color_palette`: Name of the color palette to use (e. g., `'Spectral', 'viridis', 'viridis_r', 'magma', 'magma_r'`). Defaults to 'Spectral'.

The `plot_MSV` function returns a Plotly Figure object containing the MSV metrics visualization. 

**MSV Bubble Plot** combines three visual channels in one view: each bubble’s **position**(x, (y), (z)) encodes the MSV embedding of pairwise distribution distances, sources that cluster close together share similar distributions, while those farther apart diverge; the **color** intensity maps to the Source Probabilistic Outlyingness (SPO), lighter hues indicate distributions near the central/centroid, darker hues flag highly divergent sources; and the **size** of each bubble is proportional to the number of samples in that source—larger bubbles denote more data, smaller bubbles fewer. Reading these together lets you instantly spot high-volume outliers (large, dark bubbles), low-volume anomalies (small, dark bubbles), and clusters of stable sources (grouped, light-colored bubbles).

In [ ]:
univariate_msv_plot = ds.plot_MSV(
    msv_metrics=univariate_msv,
    dimensions=1
)
univariate_msv_plot.show()

In [ ]:
multivariate_msv_plot = ds.plot_MSV(
    msv_metrics=multivariate_msv,
    dimensions=1
)
multivariate_msv_plot.show()

In [ ]:
conditional_msv_plot = ds.plot_MSV(
    msv_metrics=conditional_msv,
    dimensions=1
)
conditional_msv_plot.show()

<a id="unsupervised-interpretation"></a>
### 2.4 Interpretation of temporal changes in IGT projections

---

According to the layout of time batches in IGT projections we define the following four types of temporal changes (quoting text from our previous publication (Sáez & García-Gómez, 2018)):

- Trend:  Continuous and smooth change in the probability distributions of time batches over time, along the full-time period, or within a sub-period. Trends can be linear or, more generally, curved. Trends are represented in the IGT projection as a continuous flow of time batches through a time-related direction (Sáez & García-Gómez, 2018). Trends have been extensively explored in relation to temporal variability in electronic health records (Sáez et al., 2020) and other biomedical datasets (Andreu-Perez et al., 2015).
- Abrupt change: A sudden change in probability distributions at a specific time point, leading to a new data inherent concept which is maintained afterward. Abrupt changes are represented in the IGT projection as a gap between two groups of continuous time batches. Multiple abrupt changes can occur in a data repository, splitting the dataset into multiple clusters of time batches (see the definition of temporal subgroups). A single time batch could be abruptly separated from the rest, generally due to some specific context in its data (e.g., transient states, incomplete batches); in that case, we will talk about an outlier batch (Sáez et al., 2015). Understanding such abrupt changes is critical to addressing dataset shifts in machine learning contexts (Quiñonero-Candela, 2009; Moreno-Torres et al., 2012).
- Temporal subgroups: Conceptually related groups of time periods at which probability distributions are similar within a group but dissimilar between groups, i.e., forming clusters of time batches. Abrupt changes generally split data into temporal subgroups. A consecutive time flow between batches at two temporal subgroups would indicate a recurrent behavior. An outlier batch will not be considered within any subgroup. Temporal subgroup analysis has been shown to enhance quality control and interpretation in biomedical repositories (Sáez et al., 2016; Schlegel & Ficheur, 2017).
- Seasonality: Repetition of some change patterns at a specific time period throughout the IGT projection. Seasonality should be represented in the IGT projection as repetitive cycles over the general temporal flow. We could find local seasonality within a specific time period or global seasonality across the full study period. Global seasonality should be maintained even across multiple temporal subgroups; e.g., in a data repository partitioned into various temporal subgroups, a global yearly variation should be maintained across the different subgroups. Addressing seasonality is especially important for long-term studies where periodic trends can influence the interpretation of data variability (Sáez & García-Gómez, 2018; Sáez et al., 2015).


### 2.5 Interpretation of Source Variability patterns with MSV metrics

In the MSV simplex embedding, each data source is represented as a point (or bubble) whose color encodes its Source Probabilistic Outlyingness (SPO) and whose distance from the centroid reflects its divergence. From the spatial arrangement of sources we can distinguish four characteristic patterns of **source change** (adapted from Sáez et al., 2014):

- **Uniformity** All sources lie tightly around the centroid (low Global Probabilistic Deviation, GPD) and display low SPO (light‐colored bubbles). This indicates that the various sources share very similar probability distributions—ideal for pooled analysis.

- **Gradient drift** Sources are arranged along a continuous trajectory radiating from the centroid, with SPO gradually increasing. This “trend” of source divergence suggests a monotonic shift in data characteristics across sources—e.g., a systematic protocol change or incremental sensor bias.

- **Abrupt shift** One or a few sources appear as dark‐colored outliers far from the main cluster (high SPO), with a clear gap separating them. An abrupt shift reflects a sudden change in data generation—such as a new acquisition device or a major pre-processing update—that persists thereafter.

- **Source clusters** The embedding splits into two or more tight subgroups of sources, each internally homogeneous but distinct from one another. These clusters may correspond to different data‐collection sites, patient cohorts, or experimental batches. Identifying such subgroups can guide stratified analyses or targeted harmonization efforts.

By reading the **position**, **color**, and **size** of each bubble together, practitioners can quickly pinpoint stable sources, detect gradual drifts, flag sudden deviations, and uncover hidden subgroups in their multi‐source datasets.

<a id="unsupervised-summary"></a>
### 2.6 Summary of dashi unsupervised characterization functions

---

Table 1: Functions in **dashi** Python library

| **Input Object**                                                                                         | **Function**                            | **Output Generated**                     |
|----------------------------------------------------------------------------------------------------------|-----------------------------------------|------------------------------------------|
| `DataFarme`                                                                                              | format_data                             | `DataFrame`                              |
| `DataFrame`                                                                                              | estimate_univariate_data_temporal_map   | `DataTemporalMap`                        |
| `DataFrame`                                                                                              | estimate_multivariate_data_temporal_map | `MultivariateDateTemporalMap`            |
| `DataFrame`                                                                                              | estimate_conditional_data_temporal_map  | `Dict[str, MultiVariateDataTemporalMap]` |
| `DataFrame`                                                                                              | estimate_univariate_data_source_map     | `DataSourceMap`                          |
| `DataFrame`                                                                                              | estimate_multivariate_data_source_map   | `MultivariateDataSourceMap`              |
| `DataFrame`                                                                                              | estimate_conditional_data_source_map    | `Dict[str, MultiVariateDataSourceMap]`   |
| `DataTemporalMap` or <br/>`MultivariateDateTemporalMap` or <br/>`Dict[str, MultiVariateDataTemporalMap]` | estimate_igt_projection                 | `IGTProjection`                          |
| `DataSourceMap` or <br/>`MultiVariateDataSourceMap` or <br/>`Dict[str, MultiVariateDataSourceMap]`       | estimate_MSV_metrics                    | `MSVMetrics`                             |
| `DataTemporalMap`                                                                                        | plot_univariate_data_temporal_map       | `Figure`                                 |
| `MultivariateDateTemporalMap`                                                                            | plot_multivariate_data_temporal_map     | `Figure`                                 |
| `Dict[str, MultiVariateDataTemporalMap]`                                                                 | plot_conditional_data_temporal_map      | `List[Figure]`                           |
| `DataSourceMap`                                                                                          | plot_univariate_data_source_map         | `Figure`                                 |
| `MultivariateDataSourceMap`                                                                              | plot_multivariate_data_source_map       | `Figure`                                 |
| `Dict[str, MultiVariateDataSourceMap]`                                                                   | plot_conditional_data_source_map        | `List[Figure`                            |
| `IGTProjection`                                                                                          | plot_IGT_projection                     | `Figure`                                 |
| `MSVMetrics`                                                                                             | plot_MSV                                | `Figure`                                 |


<a id="supervised"></a>
## 3 Supervised characterization analysis

<a id="supervised-preprocessing"></a>
### 3.1 Data pre-processing

---

For the supervised characterization, data must not contain nan values. The user should decide how to deal with them. For this example, we will just drop all the rows with nan values:

In [ ]:
data_without_nan = dataset_formatted.dropna(axis=1, how='any')

<a id="supervised-analysis"></a>
### 3.2 Data analysis

---

#### 3.2.1 Multi-Batch model training and validation

The `estimate_multibatch_models` function automatically trains RandomForest based models across multiple batches (temporal or source) for both classification and regression tasks. Additionally, it validates each trained model's performance on every other batch. Requires specifying one target variable (regression or classification) and at least one numerical or categorical input feature within the input DataFrame. At the same time, it is necessary to provide either a date variable (indicating the period with the corresponding argument) or a source variable. The date variable must be a valid date, and the source variable categories need to be specified as strings. Additionally, it is recommended that the amount of data in each batching group be sufficient for statistical representativeness. This function has as input the following arguments:

- `data`: The input `DataFrame` containing both numerical and categorical features, as well as the target variables.
- `inputs_numerical_column_names`: List of column names representing categorical input features, if applicable.
- `output_regression_column_name`: Column name for the classification target variable, if applicable.
- `date_column_name`: Column name containing date or time information for temporal batching in string format, if applicable. Mandatory for temporal dataset shift characterization
- `source_column_name`: Column name representing the source of the data, if applicable. Mandatory for temporal dataset shift characterization
- `period`: Period for batching the data ('month' or 'year') when using temporal batching.
- `learning_strategy`: Defines the learning strategy: 'from_scratch' or 'cumulative'.

The `estimate_multibatch_models` returns a dictionary containing the calculated metrics for each batch and model combination
. 

The regression metrics storaged are:

- 'MEAN_ABSOLUTE_ERROR'
- 'MEAN_SQUARED_ERROR'
- 'ROOT_MEAN_SQUARED_ERROR'
- 'R_SQUARED'

The classification metrics storaged are:

- 'AUC_{class_identifier}'
- 'AUC_MACRO'
- 'LOGLOSS'
- 'RECALL_{class_identifier}'
- 'PRECISION_{class_identifier}'
- 'F1-SCORE_{class_identifier}'
- 'ACCURACY'
- 'RECALL_MACRO'
- 'RECALL_MICRO'
- 'RECALL_WEIGHTED'
- 'PRECISION_MACRO'
- 'PRECISION_MICRO'
- 'PRECISION_WEIGHTED'
- 'F1-SCORE_MACRO'
- 'F1-SCORE_MICRO'
- 'F1-SCORE_WEIGHTED'

##### Example of characterization by temporal batches:

In [ ]:
categorical_predictors = CATEGORICAL_VARIABLES.copy()
categorical_predictors.remove(LABEL_NAME)

metrics_temporal = ds.estimate_multibatch_models(data=data_without_nan,
                                                 inputs_categorical_column_names=categorical_predictors,
                                                 output_classification_column_name=LABEL_NAME,
                                                 date_column_name='FECHA_INGRESO',
                                                 period='year',
                                                 learning_strategy='cumulative',
                                                 model_type="random_forest"
                                                 )

##### Example of characterization by multi-source batches: 

In [ ]:
print(data_without_nan['SECTOR'].value_counts())
data_without_nan = data_without_nan[data_without_nan['SECTOR'] != 99]

In [ ]:
metrics_source = ds.estimate_multibatch_models(data=data_without_nan,
                                               inputs_categorical_column_names=categorical_predictors,
                                               output_classification_column_name=LABEL_NAME,
                                               source_column_name='SECTOR',
                                               period='year',
                                               learning_strategy='from_scratch',
                                               model_type="random_forest"
                                               )

<a id="supervised-visualization"></a>
### 3.3 Results display

---

#### 3.3.1 Plot models' performance metrics

The `plot_multibatch_performance` function displays a heatmap of the specified metric for multiple batches of training and test models from the metrics dictionary obtained by the `estimate_multibatch_models` function. The function takes a dictionary of metrics and filters them based on the metric identifier. It then generates a heatmap where the x-axis represents the test batches, the y-axis represents the training batches, and the color scale indicates the values of the specified metric. This function has as input the following arguments:

- `metrics`: A dictionary where keys are tuples of (training_batch, test_batch, dataset_type), and values are the metric values for the corresponding combination. The dataset_type should be 'test' to include the metric in the heatmap.
- `metric_name`: The name of the metric to visualize. The function will filter metrics based on this identifier and only plot those for the 'test' set. Regression metric names, when applicable:
    
    - 'MEAN_ABSOLUTE_ERROR'
    - 'MEAN_SQUARED_ERROR'
    - 'ROOT_MEAN_SQUARED_ERROR'
    - 'R_SQUARED'
        
    Classification metric names, when applicable:
  
    - 'AUC_{class_identifier}'
    - 'AUC_MACRO'
    - 'LOGLOSS'
    - 'RECALL_{class_identifier}'
    - 'PRECISION_{class_identifier}'
    - 'F1-SCORE_{class_identifier}'
    - 'ACCURACY'
    - 'RECALL_MACRO'
    - 'RECALL_MICRO'
    - 'RECALL_WEIGHTED'
    - 'PRECISION_MACRO'
    - 'PRECISION_MICRO'
    - 'PRECISION_WEIGHTED'
    - 'F1-SCORE_MACRO'
    - 'F1-SCORE_MICRO'
    - 'F1-SCORE_WEIGHTED'
    
This function generates and displays an interactive heatmap using Plotly, and does not return any value.

In [ ]:
recall_temporal = ds.plot_multibatch_performance(metrics=metrics_temporal,
                               metric_name='RECALL_MACRO')
recall_temporal.show()

In [ ]:
f1_temporal = ds.plot_multibatch_performance(metrics=metrics_temporal,
                               metric_name='F1-SCORE_MACRO')
f1_temporal.show()

In [ ]:
recall_source = ds.plot_multibatch_performance(metrics=metrics_source,
                               metric_name='RECALL_MACRO')
recall_source.show()

In [ ]:
f1_source = ds.plot_multibatch_performance(metrics=metrics_source,
                               metric_name='F1-SCORE_MACRO')
f1_source.show()

<a id="supervised-arrangement"></a>
### 3.4 Metrics arrangement

---

If the user wishes to store some specific metric in a DataFrame, the `arrange_performance_metrics` allows to extract a subset of metrics from the resulting dictionary into a `pandas` `DataFrame`. This function has as input the following arguments:

- `metrics`: A dictionary containing the calculated metrics for each batch and model combination resulting from the `estimate_multibatch_models` function.
- `metric_name`: The name of the metric to be selected from the metrics dictionary. The available metrics are described above.

The `arrange_performance_metrics` returns a `DataFrame` where the rows represent the combinations and the columns represent the metric values, with the index corrected for cumulative learning strategies.

In [ ]:
metrics_temporal_frame = ds.arrange_performance_metrics(
    metrics=metrics_temporal, 
    metric_name='RECALL_MACRO'
)

<a id="supervised-interpretation"></a>
### 3.5 Interpretation of the results

---

In real-world applications, machine learning models are frequently trained on specific datasets but deployed in environments where data distributions may shift over time due to factors such as temporal changes, varying data sources, or evolving user behaviors (Chen et al., 2024; Shao et al., 2024). These distribution shifts can lead to degraded or biased model predictions, undermining the reliability and fairness of deployed systems (Gama et al., 2014; Moreno-Torres et al., 2012). The `dashi` Python library addresses this challenge by enabling supervised detection of multi-temporal and multi-source dataset shifts. By analyzing how models trained on different data batches perform across new domains, `dashi` provides valuable insights into a model’s generalization capabilities. Specifically, if a model exhibits decreased performance when evaluated on alternative data batches, it may indicate that the model is biased toward the training data or that there are significant differences between the training and evaluation data distributions. This diagnostic capability is crucial for identifying and mitigating potential biases, thereby enhancing model robustness and applicability in dynamic real-world settings (Sáez et al., 2024; Wang et al., 2024). `dashi` is an important tool for people who want to keep their machine learning models performing well and fairly as data changes over time.

<a id="supervised-summary"></a>
### 3.6 Summary of dashi supervised characterization functions

---

Table 1: Functions in **dashi** Python library

| **Input Object**                         | **Function**                 | **Output Generated** |
|------------------------------------------|------------------------------|----------------------|
| `DataFarme`                              | estimate_multibatch_models   | `Dict[str, float]`   |
| `Dict[str, float]`                       | plot_multibatch_performance  | `Figure`             |
| `Dict[str, float]`                       | arrange_performance_metrics  | `DataFrame`          |

<a id="references"></a>
## References

---

Andreu-Perez, J., Poon, C. C. Y., Merrifield, R. D., Wong, S. T. C., & Yang, G.-Z. (2015). Big Data for Health. IEEE Journal of Biomedical and Health Informatics, 19(4), 1193-1208. IEEE Journal of Biomedical and Health Informatics. https://doi.org/10.1109/JBHI.2015.2450362

Chen, M., Shen, L., Fu, H., Li, Z., Sun, J., & Liu, C. (2024). Calibration of Time-Series Forecasting: Detecting and Adapting Context-Driven Distribution Shift. Proceedings of the 30th ACM SIGKDD Conference on Knowledge Discovery and Data Mining, 341-352. https://doi.org/10.1145/3637528.3671926

Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M., & Bouchachia, A. (2014). A survey on concept drift adaptation. ACM Computing Surveys, 46(4), 1-37. https://doi.org/10.1145/2523813

Gewin, V. (2016). Data sharing: An open mind on open data. Nature, 529(7584), 117-119. https://doi.org/10.1038/nj7584-117a

Moreno-Torres, J. G., Raeder, T., Alaiz-Rodríguez, R., Chawla, N. V., & Herrera, F. (2012). A unifying view on dataset shift in classification. Pattern Recognition, 45(1), 521-530. https://doi.org/10.1016/j.patcog.2011.06.019

Quiñonero-Candela, J. (Ed.). (2009). Dataset shift in machine learning. MIT Press.

Sáez, C., Ferri, P., & García-Gómez, J. M. (2024). Resilient Artificial Intelligence in Health: Synthesis and Research Agenda Toward Next-Generation Trustworthy Clinical Decision Support. Journal of Medical Internet Research, 26(1), e50295. https://doi.org/10.2196/50295

Sáez, C., & García-Gómez, J. M. (2018). Kinematics of Big Biomedical Data to characterize temporal variability and seasonality of data repositories: Functional Data Analysis of data temporal evolution over non-parametric statistical manifolds. International Journal of Medical Informatics, 119, 109-124. https://doi.org/10.1016/j.ijmedinf.2018.09.015

Sáez, C., Gutiérrez-Sacristán, A., Kohane, I., García-Gómez, J. M., & Avillach, P. (2020). EHRtemporalVariability: Delineating temporal data-set shifts in electronic health records. GigaScience, 9(8), giaa079. https://doi.org/10.1093/gigascience/giaa079

Sáez, C., Rodrigues, P. P., Gama, J., Robles, M., & García-Gómez, J. M. (2015). Probabilistic change detection and visualization methods for the assessment of temporal stability in biomedical data quality. Data Mining and Knowledge Discovery, 29(4), 950-975. https://doi.org/10.1007/s10618-014-0378-6

Sáez, C., Zurriaga, O., Pérez-Panadés, J., Melchor, I., Robles, M., & García-Gómez, J. M. (2016). Applying probabilistic temporal and multisite data quality control methods to a public health mortality registry in Spain: A systematic approach to quality control of repositories. Journal of the American Medical Informatics Association, 23(6), 1085-1095. https://doi.org/10.1093/jamia/ocw010

Schlegel, D. R., & Ficheur, G. (2017). Secondary Use of Patient Data: Review of the Literature Published in 2016. Yearbook of Medical Informatics, 26(1), 68-71. https://doi.org/10.15265/IY-2017-032

Shao, M., Li, D., Zhao, C., Wu, X., Lin, Y., & Tian, Q. (2024). Supervised Algorithmic Fairness in Distribution Shifts: A Survey (arXiv:2402.01327). arXiv. https://doi.org/10.48550/arXiv.2402.01327

Wang, Z., Bühlmann, P., & Guo, Z. (2024). Distributionally Robust Machine Learning with Multi-source Data (arXiv:2309.02211). arXiv. https://doi.org/10.48550/arXiv.2309.02211